### Avaliação de desempenho dos agentes na base de dados 


#### 1. Preparação inicial

In [ ]:
# Importando bibliotecas necessárias
import sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
# Carregando datqaset
df = pd.read_csv('../data/processed/phishing-dataset-merged.csv', keep_default_na=False)

#### 2. Classificação de e-mails a partir do Agente

In [ ]:
# Importando agente
sys.path.append('../src')
import agente

# Separando uma amostra da base de dados
amostra = df.groupby('tipo').sample(20, random_state=2026)
resultados = []

# Iterando as linhas da base de dados
for i, linha in amostra.iterrows():
    # Calculando tempo
    inicio = time.time()
    # Bloco try/execpt a fim de previnir o loop parar caso aconteça algum erro
    try:
        # Realizando a classificação dos e-mails
        v = agente.classificar(linha['subject'], linha['body'])
        previsto, justificativa = v.classificacao, v.justificativa
    except Exception as e:
        previsto, justificativa = 'erro', str(e)[:200]

    # Adicionando resultados a lista
    resultados.append({'id': i, 'real': linha['tipo'], 'previsto': previsto, 'justificativa': justificativa, 
                       'segundos': round(time.time() - inicio, 1)})

# Transformando lista em um DataFrame pandas
df_resultados = pd.DataFrame(resultados)
df_resultados.head(15)

,id,real,previsto,confianca,justificativa,segundos
0,14030,legitimo,legitimo,0,não há sinais de spam ou phishing,25.2
1,6958,legitimo,legitimo,0,o e-mail parece ser uma comunicação interna de...,17.5
2,20554,legitimo,legitimo,0,O e-mail parece ser uma comunicação normal de ...,37.1
3,25038,legitimo,legitimo,0,O e-mail parece ser uma comunicação legítima d...,22.2
4,16363,legitimo,legitimo,0,"A mensagem parece ser legítima, pois é uma com...",31.1
5,26701,legitimo,legitimo,0,O conteúdo do e-mail parece ser uma comunicaçã...,14.9
6,7085,legitimo,legitimo,0,O e-mail parece ser uma comunicação interna de...,14.5
7,30802,legitimo,phishing,0,O e-mail parece ser uma comunicação legítima e...,29.6
8,20969,legitimo,spam,0,"O e-mail contém um anúncio de propaganda, com ...",28.4
9,4342,legitimo,legitimo,0,O e-mail parece ser uma comunicação legítima d...,16.3


#### 3. Avaliação do agente


In [ ]:
# Calculando métricas de desempenho
report = classification_report(df_resultados['real'], df_resultados['previsto'])
print(report)

              precision    recall  f1-score   support

    legitimo       0.94      0.85      0.89        20
    phishing       0.76      0.95      0.84        20
        spam       0.94      0.80      0.86        20

    accuracy                           0.87        60
   macro avg       0.88      0.87      0.87        60
weighted avg       0.88      0.87      0.87        60



#### 4. Exportando resultados

In [ ]:
# Passando dados para um arquivo csv
df_resultados.to_csv('../../experimentos/classification-emails-model-llama3.2-3b.csv', index=False)